In [11]:
from pathlib import Path
from re import I
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat


class DoclingParser:

    def __init__(self):
        self.converter = DocumentConverter()

    def parse(self, file_path: Path):
        result = self.converter.convert(str(file_path))
        document = result.document
        return document

    def inspect_structure(self, document):
        for element, level in document.iterate_items():
            print("TYPE:", type(element).__name__)
            print("LABEL:", getattr(element, "label", None))
            print("LEVEL:", level)
            text = getattr(element, "text", None)
            if text:
                print("TEXT:", text[:200])
            else:
                print("TEXT: <no text attribute>")
            print("-----")

In [18]:
parser = DoclingParser()
doc = parser.parse(Path("sample/samplexlsx.xlsx"))
parser.inspect_structure(doc)

TYPE: TableItem
LABEL: table
LEVEL: 2
TEXT: <no text attribute>
-----


d:\SystemSoftware\Miniconda\envs\genai-env\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [19]:
# from pathlib import Path
# from docling.document_converter import DocumentConverter


# class DoclingParser:

#     def __init__(self):
#         self.converter = DocumentConverter()

#     def parse(self, file_path: Path):
#         result = self.converter.convert(str(file_path))
#         document = result.document

#         return document

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Iterator, Tuple


@dataclass
class _SimpleElement:
    text: str
    label_name: str

    @property
    def label(self) -> SimpleNamespace:
        return SimpleNamespace(name=self.label_name)


class _SimpleDocument:
    """Lightweight document with a Docling-compatible iterate_items API."""

    def __init__(self, elements: list[_SimpleElement]):
        self._elements = elements

    def iterate_items(self) -> Iterator[Tuple[_SimpleElement, None]]:
        for element in self._elements:
            yield element, None


class DoclingParser:
    def __init__(self) -> None:
        self.converter = None

    def parse(self, file_path: Path):
        path = Path(file_path)
        suffix = path.suffix.lower()

        # ✅ Explicit safe formats (NO native deps)
        if suffix in {".md", ".markdown", ".txt"}:
            return self._parse_markdown(path)

        docling_supported = {
            ".pdf",
            ".docx",
            ".pptx",
            ".xlsx",
            ".csv",
            ".bmp",
            ".jpg",
            ".jpeg",
            ".png",
            ".tiff",
            ".webp",
        }
        # ❗ Optional: block unsupported formats early (prevents DLL crash)
        if suffix not in docling_supported:
            raise ValueError(f"Unsupported file type: {suffix}")

        try:
            # ✅ Only now attempt docling
            converter = self._get_docling_converter()
            result = converter.convert(str(path))
            return result.document
        except Exception as e:
            # Re-wrap or log the error so you know WHICH file failed
            raise RuntimeError(f"Docling failed to parse {path.name}: {e}")

    def _parse_markdown(self, file_path: Path) -> _SimpleDocument:
        text = file_path.read_text(encoding="utf-8")
        lines = text.splitlines()

        elements: list[_SimpleElement] = []
        for raw_line in lines:
            line = raw_line.strip()
            if not line:
                continue

            if line.startswith("#"):
                label_name = "TITLE" if not elements else "SECTION_HEADER"
                elements.append(_SimpleElement(text=line, label_name=label_name))
            elif line.startswith(("- ", "* ")):
                elements.append(
                    _SimpleElement(text=line[2:].strip(), label_name="LIST_ITEM")
                )
            else:
                elements.append(_SimpleElement(text=line, label_name="TEXT"))

        return _SimpleDocument(elements)

    def _get_docling_converter(self):
        if self.converter is not None:
            return self.converter

        try:
            from docling.document_converter import DocumentConverter

        except Exception as exc:
            raise RuntimeError(
                "Docling is required for PDF/DOCX parsing but failed to load. "
                "This is likely due to missing native dependencies (DLLs) on your system."
            ) from exc

        self.converter = DocumentConverter()
        return self.converter

    def inspect_structure(self, document, max_items: int = 15):

        count = 0

        print("\n" + "#" * 100)
        print("📚 DOCUMENT STRUCTURE INSPECTION")
        print("#" * 100)

        for element, level in document.iterate_items():

            print("\n" + "=" * 100)

            # ======================================
            # 🔹 BASIC INFO
            # ======================================

            print(f"ITEM #{count + 1}")

            element_type = type(element).__name__
            print("TYPE       :", element_type)

            label = getattr(element, "label", None)

            if hasattr(label, "name"):
                label_name = label.name
            else:
                label_name = str(label)

            print("LABEL      :", label_name)

            print("LEVEL      :", level)

            # ======================================
            # 🔹 TEXT CONTENT
            # ======================================

            text = getattr(element, "text", None)

            if text and text.strip():

                clean_text = text.strip()

                print("\n📝 CONTENT:")
                print("-" * 80)
                print(clean_text[:1000])

            else:
                print("\n📝 CONTENT: <EMPTY>")

            # ======================================
            # 🔹 HEADINGS
            # ======================================

            if label_name in ["TITLE", "SECTION_HEADER"]:

                print("\n📌 DETECTED HEADING:")
                print("-" * 80)
                print(text[:300] if text else "<NO TEXT>")

            # ======================================
            # 🔹 TABLE / CSV / XLSX
            # ======================================

            if "TABLE" in label_name.upper():

                print("\n📊 TABLE DETECTED")
                print("-" * 80)

                table_data = getattr(element, "data", None)

                if table_data:
                    print(table_data)

                elif text:
                    lines = text.split("\n")

                    if lines:
                        print("🔹 POSSIBLE COLUMNS:")
                        print(lines[0][:300])

                        if len(lines) > 1:
                            print("\n🔹 SAMPLE ROW:")
                            print(lines[1][:300])

            # ======================================
            # 🔹 IMAGE / OCR CONTENT
            # ======================================

            if "IMAGE" in label_name.upper() or "PICTURE" in label_name.upper():

                print("\n🖼 IMAGE CONTENT DETECTED")
                print("-" * 80)

                caption = getattr(element, "caption", None)

                if caption:
                    print("CAPTION:")
                    print(caption)

                elif text:
                    print("OCR / DESCRIPTION:")
                    print(text[:500])

                else:
                    print("No OCR text/caption found.")

            # ======================================
            # 🔹 CAPTIONS
            # ======================================

            if "CAPTION" in label_name.upper():

                print("\n📷 CAPTION:")
                print("-" * 80)

                print(text[:500] if text else "<NO CAPTION>")

            # ======================================
            # 🔹 METADATA
            # ======================================

            metadata = getattr(element, "metadata", None)

            if metadata:

                print("\n🧠 METADATA:")
                print("-" * 80)

                print(metadata)

            count += 1

            # ======================================
            # 🔹 LIMIT
            # ======================================

            if count >= max_items:

                print("\n" + "=" * 100)
                print("... OUTPUT TRUNCATED ...")
                print("=" * 100)

                break

In [2]:
from pathlib import Path

# =========================================
# 🔹 CONFIG
# =========================================

SAMPLE_DIR = Path("sample/")

SUPPORTED_DOC_TYPES = {
    ".pdf",
    ".doc",
    ".docx",
    ".ppt",
    ".pptx",
    ".xls",
    ".xlsx",
    ".csv",
    ".md",
    ".markdown",
    ".txt",
    ".html",
    ".xhtml",
    ".tex",
    ".adoc",
    ".asciidoc",
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tiff",
    ".webp",
}

# =========================================
# 🔹 PARSER
# =========================================

parser = DoclingParser()

# =========================================
# 🔹 TEST ALL FILES
# =========================================

files = sorted(SAMPLE_DIR.iterdir())

for file_path in files:

    suffix = file_path.suffix.lower()

    if suffix not in SUPPORTED_DOC_TYPES:
        continue

    print("\n" + "#" * 100)
    print(f"📄 TESTING FILE: {file_path.name}")
    print("#" * 100)

    try:

        doc = parser.parse(file_path)

        print("\n✅ PARSE SUCCESSFUL\n")

        parser.inspect_structure(doc, max_items=10)

    except Exception as e:

        print("\n❌ PARSE FAILED")
        print(f"ERROR: {e}")

    print("\n\n")


####################################################################################################
📄 TESTING FILE: samplebmp.bmp
####################################################################################################


d:\SystemSoftware\Miniconda\envs\genai-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Input document samplecsv.csv does not match any allowed format.



✅ PARSE SUCCESSFUL


####################################################################################################
📚 DOCUMENT STRUCTURE INSPECTION
####################################################################################################




####################################################################################################
📄 TESTING FILE: samplecsv.csv
####################################################################################################

❌ PARSE FAILED
ERROR: Docling failed to parse samplecsv.csv: File format not allowed: samplecsv.csv




####################################################################################################
📄 TESTING FILE: sampledocx.docx
####################################################################################################

✅ PARSE SUCCESSFUL


####################################################################################################
📚 DOCUMENT STRUCTURE INSPECTION
##############

Input document samplewebp.webp does not match any allowed format.



✅ PARSE SUCCESSFUL


####################################################################################################
📚 DOCUMENT STRUCTURE INSPECTION
####################################################################################################




####################################################################################################
📄 TESTING FILE: samplewebp.webp
####################################################################################################

❌ PARSE FAILED
ERROR: Docling failed to parse samplewebp.webp: File format not allowed: samplewebp.webp




####################################################################################################
📄 TESTING FILE: samplexlsx.xlsx
####################################################################################################

✅ PARSE SUCCESSFUL


####################################################################################################
📚 DOCUMENT STRUCTURE INSPECTION
########

d:\SystemSoftware\Miniconda\envs\genai-env\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
